In [7]:
import MDAnalysis as mda
import plip
import nglview as nv

In [4]:
u_result = mda.Universe("results/dep.pdbqt")
u_receptor = mda.Universe("proteins/protein_h.pdb")

In [5]:
u_combined = mda.Merge(u_result.atoms, u_receptor.atoms)

In [6]:
u_combined.atoms.write("./results/complex.pdb")

/home/fynn/Projects/molecular/venv/lib64/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:777: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn("Unit cell dimensions not found. "
/home/fynn/Projects/molecular/venv/lib64/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no information for attr: 'elements' Using default value of ' '
  warnings.warn("Found no information for attr: '{}'"
/home/fynn/Projects/molecular/venv/lib64/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn("Found no information for attr: '{}'"
/home/fynn/Projects/molecular/venv/lib64/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:1201: UserWarning: Found missing chainIDs. Corresponding atoms will use value of 'X'
  warnings.warn("Found missing chainIDs."


In [8]:
nv.show_mdanalysis(u_combined)

NGLWidget()

In [12]:
import plip.structure
import plip.structure.preparation


complex = plip.structure.preparation.PDBComplex()
complex.load_pdb("./results/complex.pdb")
complex.analyze()

In [47]:
from rdkit import Chem
from rdkit.Chem import AllChem

mol = Chem.MolFromPDBFile("proteins/protein_h.pdb")

In [48]:
Chem.SanitizeMol(mol)

rdkit.Chem.rdmolops.SanitizeFlags.SANITIZE_NONE

In [51]:
import prolif as plf

protein_plf = plf.Molecule.from_rdkit(mol)

In [52]:
poses_plf = plf.sdf_supplier("./results/dep.sdf")

In [53]:
fp = plf.Fingerprint(count=True)

In [54]:
# run on your poses
fp.run_from_iterable(poses_plf, protein_plf)

  0%|          | 0/10 [00:00<?, ?it/s]

<prolif.fingerprint.Fingerprint: 9 interactions: ['Hydrophobic', 'HBAcceptor', 'HBDonor', 'Cationic', 'Anionic', 'CationPi', 'PiCation', 'PiStacking', 'VdWContact'] at 0x7f8474a9ae00>

In [70]:
pose_index=3

In [71]:
fp.plot_lignetwork(poses_plf[pose_index])

In [ ]:
view = fp.plot_3d(
    poses_plf[pose_index], protein_plf, frame=pose_index, display_all=True
)
view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [75]:
source = mda.Universe("./proteins/3dd5.pdb")
source_protein = source.select_atoms("protein")
source_ligan = source.select_atoms("resname DEP")

In [87]:
from openbabel import pybel

In [88]:
native_ligand = next(pybel.readfile("sdf", "./ligands/dep.sdf"))
native_ligand.write("pdb", "./ligands/native.pdb")

In [89]:
docked_ligand = next(pybel.readfile("sdf", "./results/dep.sdf"))
docked_ligand.write("pdb", "./results/docked.pdb")

In [110]:
from rdkit import Chem
from rdkit.Chem import AllChem


def calculate_rmsd_rdkit(native_sdf, redocking_sdf):
    mol1 = Chem.MolFromMolFile(native_sdf, removeHs=False)
    mol2 = Chem.MolFromMolFile(redocking_sdf, removeHs=False)

    if mol1 is None or mol2 is None:
        raise ValueError("Failed to load molecule(s). Check file paths and formats.")

    # Align mol2 to mol1
    AllChem.AlignMol(mol2, mol1)

    # Calculate RMSD
    rmsd = AllChem.GetBestRMS(mol1, mol2)
    return rmsd


rmsd_value = calculate_rmsd_rdkit("./ligands/dep.sdf", "./results/dep.sdf")
print(f"RMSD Redocking vs Native: {rmsd_value:.2f} Å")


RMSD Redocking vs Native: 1.32 Å


[22:19:36] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
